In [1]:
# lets Python interact with your computer’s file system and folders.
import os

In [2]:
# Shows the current folder path you are in the terminal.
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change the current working directory to the parent folder (one level up or back). 
# chdir() → change directory, "../" → go one folder up
os.chdir("../")

In [4]:
# Shows the current folder path you are in the terminal.
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

### Update/Create the Entity:

In [5]:
# Imports a tool to create simple data containers (classes for storing values).
from dataclasses import dataclass 
from pathlib import Path # Imports Path to handle file and folder paths in a clean way.

# Turns the class into a data class:
@dataclass(frozen=True) # frozen=True means values cannot be changed after setting.

# This class stores all important paths and URLs needed for the data ingestion process in a structured way.
# A configuration container for data ingestion settings:
class DataIngestionConfig: # this is not usual python class, it is data class
    # Get all information from config.yaml file:
    root_dir: Path # Main folder where all data ingestion files will be stored.
    source_url: str # Link (URL) to download the dataset.
    local_data_file: Path # Path where the downloaded file (like a zip) will be saved.
    unzip_dir: Path # Folder where extracted data will be stored after unzipping.

### Update the Configuration Manager:


In [6]:
from textsummarizer.constants import * # Import everything from constants.py
# From common.py, import only two functions:read_yaml, create_directories
from textsummarizer.utils.common import read_yaml, create_directories

In [7]:
# Create a class and define a constructor inside it. 
# Define a class with an __init__ constructor that loads and assigns values from constants,
# config.yaml, and params.yaml for use throughout the pipeline.
class ConfigurationManager: # A class that loads all configuration and parameters once and shares them across the project.
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # project settings (folders, paths)
        self.params = read_yaml(params_filepath) # model hyperparameters


        # Creates the main folder (artifacts/) where all outputs (data, models, logs) will be stored.
        create_directories([self.config.artifacts_root]) # Then create the folder specified in the config.yaml file under artifacts_root.

    
    # Prepares configuration for the data ingestion step
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion # Gets only the data ingestion section from config.

        create_directories([config.root_dir]) # Creates folder for data ingestion (like artifacts/data_ingestion).

        # Packs all required values into a structured object:root folder, dataset URL, local file path,
        #  unzip location: get all these from class defined in the entity:
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config # Sends this config to the data ingestion pipeline.


### Update the Components:

In [8]:
import os
import urllib.request as request # Used to download data from the internet (like datasets from a URL).
import zipfile # Used to extract zip files.
from textsummarizer.logging import logger 
from textsummarizer.utils.common import get_size # Utility function from common.py to check file size (helps in monitoring dataset or downloaded files).

In [9]:
import requests
import os

class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_file(self):
        url = self.config.source_url

        r = requests.get(url)

        print("Status Code:", r.status_code)
        print("Content Type:", r.headers.get("Content-Type"))

        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

        with open(self.config.local_data_file, "wb") as f:
            f.write(r.content)

In [ ]:
# This class is responsible for: downloading dataset and extracting dataset (zip file):
class DataIngestion:
    # This is the constructor of the class, It runs automatically when you create the class.
    # It receives one input: config
    # config: DataIngestionConfig :: config should be an object of type DataIngestionConfig. 
    # It contains settings like: file paths, URLs, folder names
    def __init__(self, config: DataIngestionConfig): 
        self.config = config # Save the input config inside the class. self = current object (this class instance)
                             # self.config = class variable, config = input passed from outside


    # Download file:
    # def download_file(self):
    #     if not os.path.exists(self.config.local_data_file): # If file is NOT already downloaded → download it
    #         # Downloads file from URL in config.yaml file and saves it locally. source_url → where data comes from
    #         # local_data_file → where to save it
    #         filename, headers = request.urlretrieve(
    #             url = self.config.source_url,
    #             filename = self.config.local_data_file
    #         )
    #         logger.info(f"{filename} download! with following info: \n{headers}") # Prints message that download is complete
    #     else:
    #         logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}") # Shows size of existing file using get_size() 


    # This function downloads a file from a URL, creates the required folder if it doesn’t exist, and saves the downloaded 
    # content as a local ZIP file.
    def download_file(self): # self = refers to the current object (so it can access config, variables, etc.)
        url = self.config.source_url # Gets the download link from your config file. self.config → your configuration object.
                                     # source_url → the dataset link in YAML

        r = requests.get(url) # Sends a request to the internet to download the file. requests.get() = “go fetch this file”,
                              # r = response object. Inside r: file content, status code (success/fail), headers (metadata)

        print("Status Code:", r.status_code) # Shows if download succeeded: 200:Success, 404:Not found, 403:Forbidden
        print("Content Type:", r.headers.get("Content-Type")) # Tells what kind of file you downloaded.
                                                              # application/zip → good, text/html → wrong (means you downloaded a webpage)

        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True) # Creates the folder if it does NOT exist

        with open(self.config.local_data_file, "wb") as f: # Saves the downloaded file to disk. Write binary mode
            f.write(r.content)


    # Extracts downloaded ZIP file into a folder: 
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir # Get unzip folder path
        os.makedirs(unzip_path, exist_ok=True) # Creates the folder specified in the config.yaml if it does not exist 
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref: # Opens downloaded ZIP file in read mode
            zip_ref.extractall(unzip_path) # Extracts all files into target folder


### Update the Pipelines:

In [11]:
# This code runs the full data ingestion pipeline:
# load config → get settings → download data → extract data → handle errors if anything fails.

#  # Starts error handling block. If something fails, it goes to except.
try:
    # Initialize or call class which is specified in cofiguration manager:
    config = ConfigurationManager() # Creates an object that loads configuration and parameters. loads everything
    data_ingestion_config = config.get_data_ingestion_config() # Gets all settings needed for data ingestion from function in configuration manager. gets specific settings like paths, URLs, folders.
    data_ingestion = DataIngestion(config=data_ingestion_config) # Creates a DataIngestion object and passes configuration to it. uses those settings to run the task. Call dataingestion function form components.
    data_ingestion.download_file() # Downloads dataset from the internet (if not already present).
    data_ingestion.extract_zip_file() # Extracts the downloaded ZIP file into a folder.
except Exception as e: # Catches any error that happens in the try block.
    raise e # Re-throws the error so you can see the full error message.

[2026-05-06 16:41:57,354: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-06 16:41:57,358: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-06 16:41:57,359: INFO: common: created directory at: artifacts]
[2026-05-06 16:41:57,362: INFO: common: created directory at: artifacts/data_ingestion]
Status Code: 200
Content Type: application/zip
